# CLASS BASED EXECUTION OF HYPERGRAPH PIPELINE

In [8]:
import csv
import math
import itertools
import random
import os
import numpy as np
import pandas as pd
import networkx as nx
import hypernetx as hnx
import community as comm
import statistics
from tqdm import tqdm
import gcmi

In [2]:
class FunctionalHypergraphPipeline:
    def __init__(self):
        """
        Initializes the data pipeline framework.
        Acts as the central hub tracking intermediate variables across dependent execution steps.
        """
        self.companies = []
        self.fc = []
        self.company_index = []
        self.dates = []
        
        # Shared calculation memory layer
        self.results = {}

    # ==========================================
    # BLOCK 1 & 2: DATA INGESTION & TIME SERIES
    # ==========================================
    def load_and_clean_data(self, bsi_file='BSI100.csv', dates_file='DATES_FILE.csv'):
        """
        Loads the financial asset data sheets and associated dates, slices target 
        intervals, handles null rows safely, and builds tracking index indicators.
        """
        print("⚡ Loading and processing data sheets...")
        
        y = pd.read_csv(bsi_file)
        all_cols = list(y.columns)
        if 'Dates' in all_cols:
            all_cols.remove('Dates')
        self.companies = all_cols
        
        # Raw prices extraction 
        temp_fc = [y[comp].tolist() for comp in self.companies]
        
        # FIX #2: Proper data slicing using clean array operations
        # Slice time series: keep elements from index 5043 to 5043+3145
        temp_fc = [series[5043:5043+3145] for series in temp_fc]

        # Filter out NaN elements
        clean_companies = []
        clean_fc = []
        for j in range(len(temp_fc)):
            if len(temp_fc[j]) > 1 and not math.isnan(temp_fc[j][1]):
                clean_companies.append(self.companies[j])
                clean_fc.append(temp_fc[j])
                
        self.companies = clean_companies
        self.fc = clean_fc
        self.company_index = list(range(0, len(self.companies)))
        
        # Loading date sequences
        self.dates = []
        if os.path.exists(dates_file):
            with open(dates_file, 'r') as date_f:
                csvreader = csv.reader(date_f)
                for row in csvreader:
                    self.dates.append(row)
            print(f"✅ Loaded {len(self.dates)} date rows.")
        else:
            print(f"⚠️ Warning: {dates_file} not found. Continuing without timestamp metadata.")

        print(f"📊 Active Dataset Tracked: {len(self.companies)} items prepared.")
        return self

    def get_window_log_returns(self, company_idx, window_id):
        """
        Extracts a specific rolling window slice for a target asset
        and converts its price series into a log-returns format vector.
        FIX #1: Corrected window index calculation for proper data alignment.
        """
        j = int(window_id)
        # Fixed window length calculation (200 points per window with proper offset)
        start_idx = 100 * (j - 1)  # Start position for window j
        end_idx = start_idx + 200   # 200 point window
        raw_prices = self.fc[company_idx][start_idx:end_idx]
        
        if len(raw_prices) < 2:
            raise ValueError(f"Insufficient data in window {j} for index {company_idx} to compute returns.")

        prices_array = np.array(raw_prices)
        log_returns = np.log(prices_array[1:]) - np.log(prices_array[:-1])
        return log_returns.tolist()

    # ==========================================
    # BLOCK 3: DEPENDENCY METRICS WRAPPERS
    # ==========================================
    def mutual_info(self, x, y):
        """Calculates Gaussian Copula Mutual Information via external gcmi module."""
        import gcmi
        return gcmi.gcmi_cc(x, y)

    def interaction_info(self, x, y, z):
        """Calculates 3-Variable Interaction Information via external gcmi module."""
        import gcmi
        return gcmi.gcmi_cc(x, y) - gcmi.gccmi_ccc(x, y, z)

    # ==========================================
    # BLOCK 4: PAIRWISE BASELINE PIPELINE
    # ==========================================
    def run_pairwise_pipeline(self, P1, P2, CAP, num_windows=30):
        """
        Executes the baseline 2-body network loop: calculates mutual information,
        evaluates empirical significance against null model files, and builds state variables.
        """
        print("\n🔗 Running Pairwise Network Pipeline and P-Tests...")

        self.results.update({
            'W': [], 'w': [], 'N': [], 'NTH': [], 'EVO': [], 'EVO_TH': [], 'U': [],
            'ADJ_P_LIST_ALL': [], 'ADJ_P_LIST_ALL_TH': [],
            'ADJ_P_INDEX_ALL': [], 'ADJ_P_INDEX_ALL_TH': [],
            'WEIGHTMI_ALL': [], 'WEIGHTMI_ALL_TH': []
        })

        for u in tqdm(range(1, num_windows + 1)):
            self.results['W'].append(u)
            if len(self.dates) > (100 * u):
                self.results['w'].append(self.dates[100 * u])
            else:
                self.results['w'].append(f"W_{u}")

            # Extract Mutual Information profile
            MI_set = []
            for i in range(len(P2)):
                x, y = P1[i]
                ts_x = self.get_window_log_returns(x, u)
                ts_y = self.get_window_log_returns(y, u)
                MI_set.append(self.mutual_info(ts_x, ts_y))

            # Fetch Null Model matrix records from disk
            # FIX #8: Add file existence check
            null_data = []
            null_filename = f'BSI100MI100{u}.csv'
            if not os.path.exists(null_filename):
                print(f"⚠️ Warning: {null_filename} not found. Skipping window {u}.")
                continue
            
            with open(null_filename, 'r') as file:
                for row in csv.reader(file):
                    null_data.append(row)

            # Evaluate standard p-values
            # FIX #3: Convert strings to floats for proper numerical comparison
            Count = []
            for i in range(len(MI_set)):
                count = 0
                u_val = float(MI_set[i])
                for j in range(len(null_data[i])):
                    if u_val < float(null_data[i][j]):
                        count += 1
                Count.append(count / 100)

            # Apply Significance Layer Filter (p < 0.05)
            count2, WeightMI, P_edge_index, P_edge_list = 0, [], [], []
            for o in range(len(Count)):
                if Count[o] < 0.05:
                    count2 += 1
                    WeightMI.append(MI_set[o])
                    P_edge_list.append(P2[o])
                    P_edge_index.append(P1[o])
            
            self.results['N'].append(count2)
            self.results['WEIGHTMI_ALL'].append(WeightMI)
            self.results['ADJ_P_LIST_ALL'].append(P_edge_list)
            self.results['ADJ_P_INDEX_ALL'].append(P_edge_index)

            # Compute standard network degrees
            temp_degree = []
            for edge in P_edge_list: temp_degree.extend([edge[0], edge[1]])
            self.results['EVO'].append([temp_degree.count(node) for node in CAP])
            # FIX #6: Use dynamic company count instead of hardcoded 26
            self.results['U'].append(sum(WeightMI) / (2 * len(CAP)) if WeightMI else 0)
            
            # Secondary Threshold Layer Configuration (Mean Bounds)
            meanP = sum(WeightMI) / len(WeightMI) if WeightMI else 0
            stdevP = statistics.stdev(WeightMI) if len(WeightMI) > 1 else 0

            AdjP_index, AdjP_list, Final_weightlistMI = [], [], []
            for i in range(len(WeightMI)):
                if WeightMI[i] > (meanP + 0.0 * stdevP):
                    AdjP_index.append(P_edge_index[i])
                    AdjP_list.append(P_edge_list[i])
                    Final_weightlistMI.append(WeightMI[i])

            self.results['NTH'].append(len(AdjP_index))
            self.results['WEIGHTMI_ALL_TH'].append(Final_weightlistMI)
            self.results['ADJ_P_LIST_ALL_TH'].append(AdjP_list)
            self.results['ADJ_P_INDEX_ALL_TH'].append(AdjP_index)
            
            temp_degree_th = []
            for edge in AdjP_list: temp_degree_th.extend([edge[0], edge[1]])
            self.results['EVO_TH'].append([temp_degree_th.count(node) for node in CAP])

        print("✅ Baseline network mappings saved to local class results registry.")
        return self.results

    # ==========================================
    # BLOCK 5: HYPERGRAPH ANALYSIS METRIC
    # ==========================================
    def run_hypergraph_pipeline(self, T1, T2, CAP, num_windows=30):
        """
        Calculates 3-body Interaction Information weights, applies bootstrapping 
        via in-place random dataset shuffling, and uncovers verified hyperedges.
        """
        print("\n🔺 Running Hypergraph Interaction Weight Pipeline...")
        
        self.results.update({
            'N3': [], 'N3TH': [], 'EVO3': [], 'EVO3_TH': [], 'U1': [],
            'ADJH_LIST_ALL': [], 'ADJH_INDEX_ALL': [], 'WEIGHTII_ALL_PASS': []
        })

        for w_idx in range(1, num_windows + 1):
            II_set = []
            for i in range(len(T1)):
                x, y, z = T1[i]
                ts_x = self.get_window_log_returns(x, w_idx)
                ts_y = self.get_window_log_returns(y, w_idx)
                ts_z = self.get_window_log_returns(z, w_idx)
                II_set.append(self.interaction_info(ts_x, ts_y, ts_z))

            with open(f'REAL_VALUE_II_BSE_CAP{w_idx}.csv', 'w', newline='') as fi:
                csv.writer(fi).writerows([[val] for val in II_set])

            # Isolate Synergistic configurations (II < 0)
            Synergic_edge_list, Synergic_edge_index, Synergic_edge_value = [], [], []
            for i in range(len(II_set)):
                v = float(II_set[i])
                if v < 0:
                    Synergic_edge_list.append(T2[i])
                    Synergic_edge_index.append(T1[i])
                    Synergic_edge_value.append(v)

            if len(Synergic_edge_index) == 0:
                self.results['N3'].append(0); self.results['N3TH'].append(0)
                self.results['EVO3'].append(0); self.results['EVO3_TH'].append(0)
                self.results['U1'].append(0)
                self.results['ADJH_LIST_ALL'].append([])
                self.results['ADJH_INDEX_ALL'].append([])
                self.results['WEIGHTII_ALL_PASS'].append([])
                continue

            # Bootstrapping Shuffling Engine for Null Distribution modeling
            II_total = []
            for idx_triplet in Synergic_edge_index:
                a, b, c = idx_triplet
                ts_x = self.get_window_log_returns(a, w_idx)
                ts_y = self.get_window_log_returns(b, w_idx)
                ts_z = self.get_window_log_returns(c, w_idx)
                
                II_inter = []
                for _ in range(100):
                    random.shuffle(ts_x); random.shuffle(ts_y); random.shuffle(ts_z)
                    II_inter.append(self.interaction_info(ts_x, ts_y, ts_z))
                II_total.append(II_inter)
                
            null_filename = f'BSI100II_CAP{w_idx}.csv'
            with open(null_filename, 'w', encoding='UTF8', newline='') as fpi:
                csv.writer(fpi, delimiter=',').writerows(II_total)

            data_null = []
            with open(null_filename, 'r') as file:
                for row in csv.reader(file):
                    data_null.append([float(val) for val in row])

            # Check significance thresholds
            Count = []
            for i in range(len(Synergic_edge_value)):
                count = 0
                u_val = float(Synergic_edge_value[i])
                for j in range(100):
                    if u_val > data_null[i][j]: count += 1
                Count.append(count / 100)

            # Store matching validated structures (p < 0.05)
            count2, WeightII, H_edge_index, H_edge_list = 0, [], [], []
            for o in range(len(Count)):
                if Count[o] < 0.05:
                    count2 += 1
                    WeightII.append(Synergic_edge_value[o])
                    H_edge_index.append(Synergic_edge_index[o])
                    H_edge_list.append(Synergic_edge_list[o])

            self.results['N3'].append(count2)
            # FIX #6: Use dynamic company count
            self.results['U1'].append(sum(WeightII) / (3 * len(CAP)) if WeightII else 0)
            self.results['ADJH_LIST_ALL'].append(H_edge_list)
            self.results['ADJH_INDEX_ALL'].append(H_edge_index)
            self.results['WEIGHTII_ALL_PASS'].append(WeightII)

            temp_degree = []
            for edge in H_edge_list: temp_degree.extend([edge[0], edge[1], edge[2]])
            self.results['EVO3'].append([temp_degree.count(node) for node in CAP])

            if len(WeightII) == 0:
                self.results['N3TH'].append(0); self.results['EVO3_TH'].append(0)
                continue

            # Secondary mean parameter filters
            mean_val = sum(WeightII) / len(WeightII)
            AdjH_index, AdjH_list = [], []
            for i in range(len(WeightII)):
                if WeightII[i] < mean_val:
                    AdjH_index.append(H_edge_index[i]); AdjH_list.append(H_edge_list[i])
            self.results['N3TH'].append(len(AdjH_index))

            temp_degree_th = []
            for edge in AdjH_list: temp_degree_th.extend([edge[0], edge[1], edge[2]])
            self.results['EVO3_TH'].append([temp_degree_th.count(node) for node in CAP])

        print("✅ Synergistic Hyperedges generated and mapped completely.")
        return self.results

    # ==========================================
    # BLOCK 6: VECTOR CENTRALITY EVALUATION
    # ==========================================
    def calculate_centrality_profiles(self, CAP_index, num_windows=30):
        """
        Computes normalized network-layer eigenvector centrality profiles alongside 
        specialized Hypergraph dual intersection Line-Graph structural centralities.
        """
        print("\n🧮 Computing Network and Hypergraph Centrality Profiles...")
        
        self.results['EIGEN_VECTOR_ALL'] = []
        self.results['VECTOR_CENTRALITY_ALL'] = []

        if 'ADJ_P_INDEX_ALL' not in self.results or 'ADJH_INDEX_ALL' not in self.results:
            raise ValueError("Upstream network data missing! Execute run processes first.")

        # --- Sub-Part A: Pairwise Eigenvector Centrality ---
        for w_idx in range(num_windows):
            current_edges = self.results['ADJ_P_INDEX_ALL'][w_idx]
            if len(current_edges) == 0:
                self.results['EIGEN_VECTOR_ALL'].append([])
                continue
            G2 = nx.from_edgelist(current_edges, create_using=nx.Graph)
            try:
                pairwise_vals = list(nx.eigenvector_centrality(G2, max_iter=1000).values())
                # FIX #7: Use L2 norm for proper normalization instead of sum
                l2_norm = np.sqrt(sum(v**2 for v in pairwise_vals))
                self.results['EIGEN_VECTOR_ALL'].append([v / l2_norm for v in pairwise_vals] if l2_norm > 0 else [])
            except nx.PowerIterationFailedConvergence:
                self.results['EIGEN_VECTOR_ALL'].append([0] * len(G2.nodes()))

        # --- Sub-Part B: Hypergraph Vector Centrality (Dual Line Graph) ---
        for w_idx in range(num_windows):
            hyper_idx_list = self.results['ADJH_INDEX_ALL'][w_idx]
            if len(hyper_idx_list) == 0:
                self.results['VECTOR_CENTRALITY_ALL'].append(0)
                continue

            # FIX #5: Remove premature break statement and fix line graph construction logic
            Line_symbolic = []
            for i1 in range(len(hyper_idx_list)):
                a2, b2, c2 = hyper_idx_list[i1]
                
                for j in range(len(hyper_idx_list)):
                    if i1 == j:  # Skip self-loops
                        continue
                    d2, e2, f2 = hyper_idx_list[j]
                    # Check for intersection: if any node is shared
                    if (a2 == d2 or a2 == e2 or a2 == f2 or 
                        b2 == d2 or b2 == e2 or b2 == f2 or 
                        c2 == d2 or c2 == e2 or c2 == f2):
                        Line_symbolic.append((i1, j))

            if len(Line_symbolic) == 0:
                self.results['VECTOR_CENTRALITY_ALL'].append(0)
                continue

            G1 = nx.from_edgelist(Line_symbolic, create_using=nx.Graph)
            try:
                line_vals = list(nx.eigenvector_centrality(G1, max_iter=1000).values())
                # FIX #7: Use L2 norm for consistency
                l2_norm_line = np.sqrt(sum(v**2 for v in line_vals))
                line_eigenvector_norm = [v / l2_norm_line for v in line_vals] if l2_norm_line > 0 else [0] * len(line_vals)
            except nx.PowerIterationFailedConvergence:
                line_eigenvector_norm = [0] * len(hyper_idx_list)

            # Map Line Graph metrics back onto original node tracking indexes
            vector_centrality = []
            for i3 in CAP_index:
                vc_inter = 0
                for j1 in range(len(hyper_idx_list)):
                    a, b, c = hyper_idx_list[j1]
                    if (a == i3 or b == i3 or c == i3) and j1 < len(line_eigenvector_norm):
                        vc_inter += line_eigenvector_norm[j1]
                vector_centrality.append(vc_inter / 3)
                
            self.results['VECTOR_CENTRALITY_ALL'].append(vector_centrality)

        print("✅ Dual layer structural centrality distributions completely evaluated.")
        return self.results

    # ==========================================
    # BLOCK 7: LAPLACIAN VON NEUMANN ENTROPY
    # ==========================================
    def calculate_von_neumann_entropy(self, CAP, num_nodes=None):
        """
        Synthesizes localized Graph Laplacian operators and performs spectrum trace 
        decompositions to return the structural Von Neumann Entropy trajectory.
        FIX #9: Made num_nodes dynamic based on CAP length
        """
        print("\n🌐 Computing Network Laplacian Matrices and Von Neumann Entropy...")
        
        if 'ADJ_P_LIST_ALL' not in self.results:
            raise ValueError("Baseline network profiles unavailable. Run pipeline core.")
        
        # FIX #9: Use dynamic num_nodes if not provided
        if num_nodes is None:
            num_nodes = len(CAP)
            
        num_windows = len(self.results['ADJ_P_LIST_ALL'])
        self.results.update({'ADJMP_ALL': [], 'LAPLACIAN_ALL': [], 'EIGM': [], 'EIGVM': [], 'VNE': []})

        for w_idx in range(num_windows):
            adj_matrix = [[0] * num_nodes for _ in range(num_nodes)]
            current_edges = self.results['ADJ_P_LIST_ALL'][w_idx]
            
            for edge in current_edges:
                idx_a, idx_b = CAP.index(edge[0]), CAP.index(edge[1])
                adj_matrix[idx_a][idx_b] = -1
                adj_matrix[idx_b][idx_a] = -1
            self.results['ADJMP_ALL'].append(adj_matrix)

            temp_flat_nodes = []
            for edge in current_edges: temp_flat_nodes.extend([edge[0], edge[1]])
            degree_counts = [temp_flat_nodes.count(node) for node in CAP]
            
            laplacian_matrix = [row[:] for row in adj_matrix]
            for diag_idx in range(num_nodes):
                if diag_idx < len(degree_counts):
                    laplacian_matrix[diag_idx][diag_idx] = degree_counts[diag_idx]
            self.results['LAPLACIAN_ALL'].append(laplacian_matrix)

            # Spectral Decomposition analysis
            eig_vals, eig_vecs = np.linalg.eig(np.array(laplacian_matrix))
            self.results['EIGM'].append(eig_vals)
            self.results['EIGVM'].append(eig_vecs)

            # Compute trace-normalized entropy profiles
            vne_sum, real_eigs = 0, np.real(eig_vals)
            total_eigen_sum = sum(real_eigs)
            
            if total_eigen_sum > 0:
                for val in real_eigs:
                    prob_density = val / total_eigen_sum
                    log_comp = 0 if prob_density < 0.0001 else np.log(prob_density)
                    vne_sum += (prob_density * log_comp)
            
            self.results['VNE'].append(-vne_sum)
            
        print("✅ Network Von Neumann Entropy spectral evaluations finished.")
        return self.results['VNE']

    # ==========================================
    # BLOCK 8: GEOMETRIC FORMAN-RICCI CURVATURE
    # ==========================================
    def calculate_forman_ricci_curvature(self):
        """
        Evaluates geometric curvature profiles explicitly focusing on 
        the Forman-Ricci Curvature metric for 3-body Hyperedges (FRCHE).
        """
        print("\n📐 Computing Geometric Forman-Ricci Curvature for Hyperedges...")

        if 'ADJH_INDEX_ALL' not in self.results or 'WEIGHTII_ALL_PASS' not in self.results:
            raise ValueError("Upstream hypergraph structural arrays unavailable.")

        num_windows = len(self.results['ADJH_INDEX_ALL'])
        self.results.update({'FRlistH': [], 'FRH': []})

        for w_idx in range(num_windows):
            H_edge_index = self.results['ADJH_INDEX_ALL'][w_idx]
            WeightII = self.results['WEIGHTII_ALL_PASS'][w_idx]
            num_hyperedges = len(H_edge_index)

            if num_hyperedges == 0:
                self.results['FRlistH'].append([]); self.results['FRH'].append(0)
                continue

            FRCHE = []
            for i in range(num_hyperedges):
                a, b, c = H_edge_index[i]
                We = abs(WeightII[i])
                
                Wi, Wj, Wk = 0, 0, 0
                sigma1, sigma2, sigma3 = 0, 0, 0
                
                for j in range(num_hyperedges):
                    d, e, f = H_edge_index[j]
                    if d == a: Wi += abs(WeightII[j])
                    if e == b: Wj += abs(WeightII[j])
                    if f == c: Wk += abs(WeightII[j])
                
                Wi -= We; Wj -= We; Wk -= We

                for k in range(num_hyperedges):
                    g, h, l = H_edge_index[k]
                    denom = np.sqrt(We * abs(WeightII[k]))
                    
                    if denom > 0:
                        if g == a and (h != b or l != c): sigma1 += (Wi / denom)
                        if h == b and (g != a or l != c): sigma2 += (Wj / denom)
                        if l == c and (g != a or h != b): sigma3 += (Wk / denom)

                FRCHE.append((Wi + Wj + Wk) - We * (sigma1 + sigma2 + sigma3))
                
            self.results['FRlistH'].append(FRCHE)
            self.results['FRH'].append(sum(FRCHE) / len(FRCHE) if FRCHE else 0)
            
        print("✅ Geometric Forman-Ricci Curvature vector sequences computed completely.")
        return self.results['FRH']

# CALCUALATION OF COMBINATIONS AND SELCTION OF STOCKS

In [11]:
import numpy as np
import networkx as nx
import gcmi
import hypernetx as hnx
from turtle import width
import csv
import itertools
import matplotlib.pyplot as plt
import random
import pandas as pd
from pandas import *
from tabulate import tabulate
import math
import community as comm
from tqdm import tqdm
#Reading the whole data from file
y=read_csv('BSI100.csv')
#Reading the companies
Companies=[]

for row in y:
    Companies.append(row)
Companies.remove('Dates')


Company_index=list(range(0,len(Companies)))



#LOADING THE DATES
DATE=[]
with open("DATES_FILE.csv", 'r') as date:
  csvreader = csv.reader(date)
  for row in csvreader:
      DATE.append(row)

CAP=['RIL IB Equity','TCS IB Equity', 'HDFC IB Equity','INFO IB Equity','HUVR IB Equity','ICICIBC IB Equity','SBIN IB Equity','ITC IB Equity','BHARTI IB Equity','BJFIN IB Equity','KMB IB Equity','HINDUNILVR IB Equity','MARUTI IB Equity','BAJAJ-AUTO IB Equity','ASIANPAINT IB Equity','DRREDDY IB Equity','SUNPHARMA IB Equity','TECHM IB Equity','LTIM IB Equity','WIPRO IB Equity','INFY IB Equity','AXISBANK IB Equity','KOTAK IB Equity','HDFCBANK IB Equity','LANDT IB Equity','JSWSTEEL IB Equity']

CAP_index=[]
CAP_values=[]

for i in range(len(CAP)):
    for j in range(len(Companies)):
        if CAP[i]==Companies[j]:
            CAP_index.append(j)


#print((CAP_values[18]))    

#Pairwise network from the selected companies

P1=list(itertools.combinations(CAP_index,2))
P2=list(itertools.combinations(CAP,2))

#Higher order network from selected companies

T1=list(itertools.combinations(CAP_index,3))
T2=list(itertools.combinations(CAP,3))

print(len(T2),len(P2),len(CAP))

6545 595 35


In [12]:
# Cell 1: Setup and Data Pre-Processing
pipeline = FunctionalHypergraphPipeline()
pipeline.load_and_clean_data(bsi_file='BSI100.csv', dates_file='DATES_FILE.csv')

# Cell 2: Run Baseline Pairwise Combinations Matrix and P-Tests
# (Ensure your external arrays P1, P2, and CAP are declared above)
pipeline.run_pairwise_pipeline(P1=P1, P2=P2, CAP=CAP, num_windows=30)

# Cell 3: Run Triplet Hypergraph Interaction Weight Extraction Loops
# (Ensure your external arrays T1, T2, and CAP are declared above)
pipeline.run_hypergraph_pipeline(T1=T1, T2=T2, CAP=CAP, num_windows=30)

# Cell 4: Quantity Analytics Step 1 - Centrality Target Mappings
pipeline.calculate_centrality_profiles(CAP_index=CAP_index, num_windows=30)

# Cell 5: Quantity Analytics Step 2 - Spectral Von Neumann Entropy Vector
# FIX #9: Pass CAP to let function determine num_nodes dynamically
vne_scores = pipeline.calculate_von_neumann_entropy(CAP=CAP)

# Cell 6: Quantity Analytics Step 3 - Geometric Forman-Ricci Curvature Spectrum
fr_scores = pipeline.calculate_forman_ricci_curvature()

⚡ Loading and processing data sheets...
✅ Loaded 3145 date rows.
📊 Active Dataset Tracked: 86 items prepared.

🔗 Running Pairwise Network Pipeline and P-Tests...
